In [1]:
import pandas as pd
import re
import spacy
import nltk
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer
nltk.download("stopwords")
from nltk.corpus import stopwords

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\47462\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [2]:
texts = pd.read_csv("../data/raw/question_texts_texts.csv")
texts.head()

,sanity_text_id,serialNumber,title,body
0,01ff8ad2-ef5b-4cf2-8f1d-b44663790fa4,HT_105_Ørner,Ørner,## Ørner\r\n\r\n![Ørn med fisk i klørne. Foto....
1,0202b907-4516-4beb-8e07-759e473b5ea2,HT_43_Astronaut,Astronaut,## Astronaut\r\n\r\n![Astronaut viser tommel o...
2,02df17fa-3bfa-4f87-8cd5-d523c12c9a1f,HT_04_Snork,Snork,## Snork\r\n\r\n– Jeg vil ikke sove på rommet ...
3,035af603-c895-4f40-a3e8-4b05a07a09bd,HT_125_Miniatyrer-William,Miniatyrer - gjennom nåløyet,## Miniatyrer - gjennom nåløyet\r\n\r\n![Forsi...
4,05c18c0e-b6b5-40b8-9009-130b044693c6,HT_135_NRK,NRK,## NRK\r\n\r\n![NRK-bygget Marienlyst i Oslo](...


In [4]:
texts["title"] = texts["title"].fillna("").astype(str)
texts["body"] = texts["body"].fillna("").astype(str)


# Show a few rows of the important text columns
texts[["title", "body"]].head()

,title,body
0,Ørner,## Ørner\r\n\r\n![Ørn med fisk i klørne. Foto....
1,Astronaut,## Astronaut\r\n\r\n![Astronaut viser tommel o...
2,Snork,## Snork\r\n\r\n– Jeg vil ikke sove på rommet ...
3,Miniatyrer - gjennom nåløyet,## Miniatyrer - gjennom nåløyet\r\n\r\n![Forsi...
4,NRK,## NRK\r\n\r\n![NRK-bygget Marienlyst i Oslo](...


In [5]:
# Load spaCy's Norwegian Bokmål small model
nlp = spacy.load("nb_core_news_sm")

print("spaCy Norwegian model loaded successfully.")

spaCy Norwegian model loaded successfully.


In [7]:
# Load Norwegian stopwords from NLTK
norwegian_stopwords = set(stopwords.words("norwegian"))

# These are common function words that are usually not useful for topic modeling
custom_stopwords = {
    "og", "i", "på", "av", "til", "for", "er", "som", "det",
    "de", "en", "et", "å", "den", "med", "har", "om", "ikke"
}

# Merge both sets into one final stopword list
all_stopwords = norwegian_stopwords.union(custom_stopwords)

print("Number of stopwords:", len(all_stopwords))
print(sorted(all_stopwords)[:30])

Number of stopwords: 172
['alle', 'at', 'av', 'bare', 'begge', 'ble', 'blei', 'bli', 'blir', 'blitt', 'både', 'båe', 'da', 'de', 'deg', 'dei', 'deim', 'deira', 'deires', 'dem', 'den', 'denne', 'der', 'dere', 'deres', 'det', 'dette', 'di', 'din', 'disse']


In [8]:
# This function does light cleaning before NLP processing
def clean_text(text):
    # Convert text to lowercase
    text = text.lower()
    
    # Replace multiple spaces/newlines/tabs with a single space
    text = re.sub(r"\s+", " ", text)
    
    # Remove punctuation and special symbols
    # Keep word characters, spaces, and Norwegian letters
    text = re.sub(r"[^\w\sæøå]", " ", text)
    
    # Remove spaces from beginning and end
    text = text.strip()
    
    return text

In [9]:
# Extract words from text using regex
# \b\w+\b means: find word-like sequences
def get_words(text):
    return re.findall(r"\b\w+\b", text.lower())

# Count total number of words
def word_count(text):
    return len(get_words(text))

# Count total number of characters, including spaces and punctuation
def char_count(text):
    return len(text)

# Count total number of characters excluding spaces
def char_no_space_count(text):
    return len(text.replace(" ", ""))

# Count sentences by splitting on ., !, or ?
# This is a simple approximation, not perfect sentence detection
def sentence_count(text):
    sentences = re.split(r"[.!?]+", text)
    return len([s for s in sentences if s.strip()])

# Compute average word length
def avg_word_length(text):
    words = get_words(text)
    if not words:
        return 0
    return sum(len(word) for word in words) / len(words)

# Count how many unique words exist in the text
def unique_word_count(text):
    return len(set(get_words(text)))

# Type-token ratio = unique words / total words
# This is a simple measure of lexical diversity
def type_token_ratio(text):
    words = get_words(text)
    if not words:
        return 0
    return len(set(words)) / len(words)

In [10]:
# This function:
# 1. cleans the text
# 2. processes it with spaCy
# 3. removes punctuation, numbers, stopwords
# 4. converts words to lemmas
# 5. returns a list of cleaned tokens

def preprocess_norwegian(text):
    # Basic cleaning first
    text = clean_text(text)
    
    # Run the Norwegian spaCy model
    doc = nlp(text)

    tokens = []
    
    for token in doc:
        # Convert each token to its lemma (base form)
        lemma = token.lemma_.lower().strip()

        # Skip spaces
        if token.is_space:
            continue
        
        # Skip punctuation
        if token.is_punct:
            continue
        
        # Skip numbers
        if token.like_num:
            continue
        
        # Skip empty lemmas
        if not lemma:
            continue
        
        # Skip stopwords
        if lemma in all_stopwords:
            continue
        
        # Skip very short tokens
        if len(lemma) < 2:
            continue

        tokens.append(lemma)

    return tokens

In [11]:
# This version keeps only content-heavy parts of speech
# For topic modeling, nouns, adjectives, verbs, and proper nouns are often more useful

def preprocess_norwegian_keep_pos(text, allowed_pos={"NOUN", "ADJ", "VERB", "PROPN"}):
    # Basic cleaning first
    text = clean_text(text)
    
    # Run the Norwegian spaCy model
    doc = nlp(text)

    tokens = []
    
    for token in doc:
        # Convert token to lemma
        lemma = token.lemma_.lower().strip()

        # Keep only selected parts of speech
        if token.pos_ not in allowed_pos:
            continue
        
        # Skip spaces
        if token.is_space:
            continue
        
        # Skip punctuation
        if token.is_punct:
            continue
        
        # Skip numbers
        if token.like_num:
            continue
        
        # Skip empty lemmas
        if not lemma:
            continue
        
        # Skip stopwords
        if lemma in all_stopwords:
            continue
        
        # Skip very short tokens
        if len(lemma) < 2:
            continue

        tokens.append(lemma)

    return tokens

In [15]:
# Create a basic cleaned version of the full text
texts["clean_text"] = texts["body"].apply(clean_text)

# Create a tokenized + lemmatized version
texts["tokens_nb"] = texts["body"].apply(preprocess_norwegian)

# Create a more topic-focused token version
texts["tokens_nb_topic"] = texts["body"].apply(preprocess_norwegian_keep_pos)

# Join the token lists back into strings
# This is useful for vectorizers such as CountVectorizer or TF-IDF
texts["processed_text"] = texts["tokens_nb"].apply(lambda x: " ".join(x))
texts["processed_text_topic"] = texts["tokens_nb_topic"].apply(lambda x: " ".join(x))

# Inspect the results
texts[["body", "processed_text", "processed_text_topic"]].head()

,body,processed_text,processed_text_topic
0,## Ørner\r\n\r\n![Ørn med fisk i klørne. Foto....,ørne ørn fisk klør foto ørn ørne felles navn u...,ørne ørn fisk klør foto ørn ørne felles navn u...
1,## Astronaut\r\n\r\n![Astronaut viser tommel o...,astronau astronau vise tommel mens sitte fasts...,astronau astronau vise tommel sitte fastspent ...
2,## Snork\r\n\r\n– Jeg vil ikke sove på rommet ...,snork sove rom eivind irma sitte utenfor hytte...,snork sove rom eivind irma sitte hytte spise f...
3,## Miniatyrer - gjennom nåløyet\r\n\r\n![Forsi...,miniatyre gjennom nåløye forside willard wigan...,miniatyre nåløye forside willard instagramside...
4,## NRK\r\n\r\n![NRK-bygget Marienlyst i Oslo](...,nrk nrk bygge marienlyst oslo nrk norge stor m...,nrk nrk bygge marienlyst oslo nrk norge stor m...


In [19]:
# Basic text length statistics
texts["title_word_count"] = texts["title"].apply(word_count)
texts["body_word_count"] = texts["body"].apply(word_count)

# Character-based statistics
texts["title_char_count"] = texts["title"].apply(char_count)
texts["body_char_count"] = texts["body"].apply(char_count)

# More detailed body statistics
texts["body_char_no_space_count"] = texts["body"].apply(char_no_space_count)
texts["body_sentence_count"] = texts["body"].apply(sentence_count)
texts["body_avg_word_length"] = texts["body"].apply(avg_word_length)
texts["body_unique_word_count"] = texts["body"].apply(unique_word_count)
texts["body_type_token_ratio"] = texts["body"].apply(type_token_ratio)

# Token counts after Norwegian preprocessing
texts["processed_token_count"] = texts["tokens_nb"].apply(len)
texts["processed_topic_token_count"] = texts["tokens_nb_topic"].apply(len)

# Show the updated DataFrame
texts.head()

,sanity_text_id,serialNumber,title,body,clean_text,tokens_nb,tokens_nb_topic,processed_text,processed_text_topic,title_word_count,body_word_count,title_char_count,body_char_count,body_char_no_space_count,body_sentence_count,body_avg_word_length,body_unique_word_count,body_type_token_ratio,processed_token_count,processed_topic_token_count
0,01ff8ad2-ef5b-4cf2-8f1d-b44663790fa4,HT_105_Ørner,Ørner,## Ørner\r\n\r\n![Ørn med fisk i klørne. Foto....,ørner ørn med fisk i klørne foto ør...,"[ørne, ørn, fisk, klør, foto, ørn, ørne, felle...","[ørne, ørn, fisk, klør, foto, ørn, ørne, felle...",ørne ørn fisk klør foto ørn ørne felles navn u...,ørne ørn fisk klør foto ørn ørne felles navn u...,1,251,5,1532,1294,24,4.713147,138,0.549801,154,142
1,0202b907-4516-4beb-8e07-759e473b5ea2,HT_43_Astronaut,Astronaut,## Astronaut\r\n\r\n![Astronaut viser tommel o...,astronaut astronaut viser tommel opp mens ha...,"[astronau, astronau, vise, tommel, mens, sitte...","[astronau, astronau, vise, tommel, sitte, fast...",astronau astronau vise tommel mens sitte fasts...,astronau astronau vise tommel sitte fastspent ...,1,449,9,2428,1990,31,4.211581,229,0.510022,215,175
2,02df17fa-3bfa-4f87-8cd5-d523c12c9a1f,HT_04_Snork,Snork,## Snork\r\n\r\n– Jeg vil ikke sove på rommet ...,snork jeg vil ikke sove på rommet med eivind...,"[snork, sove, rom, eivind, irma, sitte, utenfo...","[snork, sove, rom, eivind, irma, sitte, hytte,...",snork sove rom eivind irma sitte utenfor hytte...,snork sove rom eivind irma sitte hytte spise f...,1,423,5,2510,2088,71,4.177305,192,0.453901,217,199
3,035af603-c895-4f40-a3e8-4b05a07a09bd,HT_125_Miniatyrer-William,Miniatyrer - gjennom nåløyet,## Miniatyrer - gjennom nåløyet\r\n\r\n![Forsi...,miniatyrer gjennom nåløyet forsiden av wil...,"[miniatyre, gjennom, nåløye, forside, willard,...","[miniatyre, nåløye, forside, willard, instagra...",miniatyre gjennom nåløye forside willard wigan...,miniatyre nåløye forside willard instagramside...,3,243,28,1349,1108,18,4.329218,141,0.580247,110,99
4,05c18c0e-b6b5-40b8-9009-130b044693c6,HT_135_NRK,NRK,## NRK\r\n\r\n![NRK-bygget Marienlyst i Oslo](...,nrk nrk bygget marienlyst i oslo nrk er n...,"[nrk, nrk, bygge, marienlyst, oslo, nrk, norge...","[nrk, nrk, bygge, marienlyst, oslo, nrk, norge...",nrk nrk bygge marienlyst oslo nrk norge stor m...,nrk nrk bygge marienlyst oslo nrk norge stor m...,1,483,3,2777,2325,47,4.403727,206,0.426501,268,251


In [21]:
# Create a summary dictionary containing overall dataset statistics
summary = {
    "rows": len(texts),
    "avg_body_word_count": texts["body_word_count"].mean(),
    "avg_body_char_count": texts["body_char_count"].mean(),
    "avg_body_sentence_count": texts["body_sentence_count"].mean(),
    "avg_body_avg_word_length": texts["body_avg_word_length"].mean(),
    "avg_body_unique_word_count": texts["body_unique_word_count"].mean(),
    "avg_body_type_token_ratio": texts["body_type_token_ratio"].mean(),
    "avg_processed_token_count": texts["processed_token_count"].mean(),
    "max_body_word_count": texts["body_word_count"].max(),
    "min_body_word_count": texts["body_word_count"].min(),
}

# Convert the summary into a DataFrame for easy display
summary_texts = pd.DataFrame(summary.items(), columns=["Metric", "Value"])

# Show summary
summary_texts

,Metric,Value
0,rows,159.000000
1,avg_body_word_count,355.503145
2,avg_body_char_count,2050.226415
3,avg_body_sentence_count,32.440252
4,avg_body_avg_word_length,4.499078
5,avg_body_unique_word_count,178.144654
6,avg_body_type_token_ratio,0.536831
7,avg_processed_token_count,182.622642
8,max_body_word_count,1011.000000
9,min_body_word_count,21.000000


In [22]:
# Show standard statistics such as mean, std, min, max, quartiles
texts[[
    "title_word_count",
    "body_word_count",
    "body_char_count",
    "body_sentence_count",
    "body_avg_word_length",
    "body_unique_word_count",
    "body_type_token_ratio",
    "processed_token_count",
    "processed_topic_token_count"
]].describe()

,title_word_count,body_word_count,body_char_count,body_sentence_count,body_avg_word_length,body_unique_word_count,body_type_token_ratio,processed_token_count,processed_topic_token_count
count,159.000000,159.000000,159.000000,159.000000,159.000000,159.000000,159.000000,159.000000,159.000000
mean,2.547170,355.503145,2050.226415,32.440252,4.499078,178.144654,0.536831,182.622642,164.628931
std,2.030458,178.821702,985.597826,17.115613,0.379770,73.100749,0.091021,88.712998,79.642759
min,1.000000,21.000000,159.000000,3.000000,3.658824,15.000000,0.341270,10.000000,9.000000
25%,1.000000,218.000000,1297.000000,18.000000,4.228158,131.000000,0.470208,113.000000,102.000000
50%,2.000000,369.000000,2140.000000,32.000000,4.512397,192.000000,0.528708,195.000000,174.000000
75%,4.000000,460.000000,2686.500000,47.000000,4.720130,221.000000,0.589574,240.000000,215.000000
max,9.000000,1011.000000,5492.000000,77.000000,5.982759,450.000000,0.800000,502.000000,440.000000


In [23]:
# Flatten all processed token lists into one long list
all_tokens = [token for row in texts["tokens_nb"] for token in row]

# Count token frequency
token_freq = Counter(all_tokens)

# Convert the 50 most common tokens into a DataFrame
common_tokens_df = pd.DataFrame(token_freq.most_common(50), columns=["token", "count"])

# Show top 20
common_tokens_df.head(20)

,token,count
0,få,276
1,mye,240
2,se,234
3,stor,216
4,annen,198
5,hel,175
6,gå,171
7,foto,160
8,komme,156
9,norge,153


In [24]:
# Flatten the topic-focused token lists
all_topic_tokens = [token for row in texts["tokens_nb_topic"] for token in row]

# Count frequency
topic_token_freq = Counter(all_topic_tokens)

# Convert top results into DataFrame
common_topic_tokens_df = pd.DataFrame(topic_token_freq.most_common(50), columns=["token", "count"])

# Show top 20
common_topic_tokens_df.head(20)

,token,count
0,mye,240
1,se,232
2,få,229
3,stor,216
4,hel,175
5,gå,171
6,foto,160
7,komme,156
8,norge,153
9,liten,152


In [25]:
# Build a bigram vectorizer
# ngram_range=(2,2) means: only 2-word combinations
# min_df=2 means: only keep bigrams appearing in at least 2 documents
vectorizer = CountVectorizer(
    ngram_range=(2, 2),
    min_df=2
)

# Fit and transform the processed topic-focused text
X_bigrams = vectorizer.fit_transform(texts["processed_text_topic"])

# Sum counts of each bigram across all documents
bigram_counts = X_bigrams.sum(axis=0).A1
bigram_names = vectorizer.get_feature_names_out()

# Build a DataFrame of bigrams and their counts
bigrams_df = pd.DataFrame({
    "bigram": bigram_names,
    "count": bigram_counts
}).sort_values("count", ascending=False)

# Show top 30 bigrams
bigrams_df.head(30)

,bigram,count
1573,år gammel,19
936,nemi blad,14
1559,walt disney,13
496,hel verden,13
727,lang tid,12
731,legge egg,11
1491,veie kilo,11
491,hel tid,11
900,mye kjent,11
909,mye mye,10


In [26]:
# Remove rows where the processed topic token count is too small
# This is useful because very short texts often add noise to topic modeling
texts_filtered = texts[texts["processed_topic_token_count"] >= 5].copy()

print("Original number of rows:", len(texts))
print("Filtered number of rows:", len(texts_filtered))

Original number of rows: 159
Filtered number of rows: 159


In [27]:
# Save the processed DataFrame to a new CSV file
texts.to_csv("../data/processed/question_texts_preprocessed.csv", index=False)

print("Processed file saved successfully.")

Processed file saved successfully.
